# ML05 · KMeans 分群

用最小範例建立對非監督分群的直覺：讓 KMeans 自己把沒有標籤的資料分成幾群，並學會用肘部法挑選群數。

## 學習目標
- 理解分群（clustering）與非監督學習（unsupervised learning）的定位：沒有標準答案，靠資料結構自己分組。
- 建立 KMeans 的直覺：質心（centroid）與「分配點、移動質心」反覆迭代的過程。
- 會用 sklearn 的 KMeans，掌握 fit_predict、labels_、cluster_centers_ 三件事。
- 認識慣性（inertia）的意義，並用肘部法（elbow method）決定群數 k。
- 能把分群結果依群別上色，做出可解讀的視覺化。

In [ ]:
# 概念：畫圖前的共用設定，讓中文標題正常顯示、固定亂數種子方便對照
import numpy as np
import matplotlib.pyplot as plt

# 讓 matplotlib 能顯示繁體中文，避免標題出現方框
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "DFKai-SB", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False   # 負號用 ASCII，避免顯示異常

print("環境設定完成")

## 什麼是分群與非監督學習

分群（clustering）是把一堆資料依「彼此有多像」自動分成幾組（群，cluster）的工作。它屬於非監督學習（unsupervised learning）：訓練資料只有特徵、沒有標籤（label），模型要自己從資料結構找出分組。

為什麼要學：前面的分類（classification）是監督學習（supervised learning），需要每筆資料都附正確答案；但真實情境常常沒有答案，例如想把一批街區依型態分組卻沒人事先標好。這時就靠分群讓資料自己說話。

差別一句話：監督學習是「給答案、學對應」；非監督學習是「沒答案、找結構」。

In [ ]:
# 概念：非監督資料只有特徵、沒有標籤；肉眼能看出幾團，正是分群想自動做到的事
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # 固定亂數種子，讓每次執行的散點一致

# 造三團二維假資料：每團在不同中心附近散開（例如三種型態的街區）
center_a = rng.normal(loc=[2, 2], scale=0.5, size=(30, 2))
center_b = rng.normal(loc=[8, 3], scale=0.5, size=(30, 2))
center_c = rng.normal(loc=[5, 8], scale=0.5, size=(30, 2))

points = np.vstack([center_a, center_b, center_c])   # 三團疊成一份資料，共 90 個點

# 注意：此處刻意不保留各點來自哪一團的資訊，模擬「沒有標籤」的真實情境
print("資料 shape:", points.shape)   # (90, 2)：90 個樣本、每個 2 維

plt.figure(figsize=(5, 4))
plt.scatter(points[:, 0], points[:, 1], color="gray")   # 全部同色，因為我們還不知道分組
plt.title("沒有標籤的二維點：肉眼看似三團")
plt.xlabel("特徵 1")
plt.ylabel("特徵 2")
plt.show()

## KMeans 直覺：質心與迭代

KMeans 的核心是質心（centroid）：每一群用一個中心點代表。演算法不靠複雜公式，而是反覆做兩件事直到穩定：
1. 分配：把每個點分到「離它最近的質心」那一群。
2. 更新：把每群的質心移到該群所有點的平均位置。

為什麼會收斂（convergence）：每次迭代（iteration）質心都往群的中央靠，群內距離越來越小，移動幅度越來越小，最後幾乎不動就停。理解這個循環，就能明白 KMeans 在做的只是「反覆逼近群中心」。

In [ ]:
# 概念：手動跑幾輪「分配點、更新質心」，觀察質心如何從隨機位置移到群中央
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)

# 造兩團容易分開的點當示範
group1 = rng.normal(loc=[2, 2], scale=0.6, size=(40, 2))
group2 = rng.normal(loc=[7, 7], scale=0.6, size=(40, 2))
points = np.vstack([group1, group2])

# 隨機挑兩個起始質心（故意放在不太準的位置，看它怎麼移動）
centroids = np.array([[1.0, 6.0], [6.0, 2.0]])

for step in range(3):                                   # 只跑前三輪，足以看出趨勢
    # 分配：算每個點到兩個質心的距離，取較近的那群（0 或 1）
    dist = np.linalg.norm(points[:, None, :] - centroids[None, :, :], axis=2)
    assign = dist.argmin(axis=1)                        # 每個點所屬的群編號

    # 更新：把每個質心移到該群所有點的平均位置
    new_centroids = np.array([points[assign == k].mean(axis=0) for k in range(2)])

    print(f"第 {step+1} 輪後質心:\n", np.round(new_centroids, 2))
    centroids = new_centroids   # 用更新後的質心進入下一輪

plt.figure(figsize=(5, 4))
plt.scatter(points[:, 0], points[:, 1], c=assign, cmap="viridis")
plt.scatter(centroids[:, 0], centroids[:, 1], color="red", marker="X", s=200)   # 紅叉為最後質心
plt.title("手動迭代：質心移到群中央")
plt.show()

## 動手用 sklearn KMeans

實務上不必自己寫迭代，sklearn 的 KMeans 物件幫我們處理好。使用時設定群數 n_clusters，再讓它從資料學出分群。

三個常用輸出：
- fit_predict：一次完成「學習 + 回傳每個點的群編號」。
- labels_：每個樣本被分到的群編號（與輸入資料一一對應）。
- cluster_centers_：各群質心的座標。

形狀：`KMeans(n_clusters=群數).fit_predict(資料)`。

In [ ]:
# 概念：用 sklearn KMeans 對多團資料分群，取得 labels_ 與 cluster_centers_
import numpy as np
from sklearn.cluster import KMeans

rng = np.random.default_rng(0)

# 造三團二維資料（與前面同樣的造法，確保自足）
center_a = rng.normal(loc=[2, 2], scale=0.5, size=(30, 2))
center_b = rng.normal(loc=[8, 3], scale=0.5, size=(30, 2))
center_c = rng.normal(loc=[5, 8], scale=0.5, size=(30, 2))
points = np.vstack([center_a, center_b, center_c])

# n_init 設明確值避免版本差異的警告；random_state 固定讓結果可重現
kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = kmeans.fit_predict(points)   # 一步到位：學習並回傳每個點的群編號

print("前 10 個點的群編號:", labels[:10])
print("labels_ 是否等於 fit_predict 回傳值:", np.array_equal(labels, kmeans.labels_))
print("三個群的質心座標:\n", np.round(kmeans.cluster_centers_, 2))
# 與肉眼預期對照：三個質心應分別落在 (2,2)、(8,3)、(5,8) 附近
print("各群點數:", np.bincount(labels))

## 選幾群才對：inertia 與肘部法

群數 k 要自己決定，而 KMeans 用慣性（inertia）衡量分群的緊密程度：它是「每個點到所屬質心距離平方的總和」，越小代表群內越聚集。

問題是：k 越大 inertia 一定越小（極端情況每個點自成一群、inertia 為 0），所以不能只看誰小。肘部法（elbow method）的做法是畫出 inertia 對 k 的曲線，找曲線從「陡降」轉為「平緩」的轉折點（手肘），那個 k 通常就是合理群數。

慣性可由 `kmeans.inertia_` 取得。

In [ ]:
# 概念：對 k 從 1 到 8 各跑一次 KMeans，蒐集 inertia 畫肘部法曲線
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

rng = np.random.default_rng(0)
center_a = rng.normal(loc=[2, 2], scale=0.5, size=(30, 2))
center_b = rng.normal(loc=[8, 3], scale=0.5, size=(30, 2))
center_c = rng.normal(loc=[5, 8], scale=0.5, size=(30, 2))
points = np.vstack([center_a, center_b, center_c])

k_values = range(1, 9)
inertias = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    km.fit(points)                  # 這裡只需要學習以取得 inertia_，不必拿回 labels
    inertias.append(km.inertia_)    # 累積每個 k 的群內離散程度

print("各 k 的 inertia:", np.round(inertias, 1))

plt.figure(figsize=(5, 4))
plt.plot(list(k_values), inertias, marker="o")
plt.title("肘部法：inertia 隨 k 變化")
plt.xlabel("群數 k")
plt.ylabel("inertia（群內離散）")
# 技巧：手肘大約在曲線「明顯轉平」的 k；本例資料有三團，預期手肘落在 k=3
plt.show()

## 分群結果視覺化與解讀

分群沒有標準答案，所以視覺化是檢查結果合不合理的主要手段（視覺檢查，visual sanity check）。做法是把點依群別上色（color by label），再把質心疊上去。

看兩件事：
- 群是否分得開：不同顏色的點有沒有明顯界線，還是混在一起。
- 質心是否落在群中央：紅色標記應該位於各色點團的中心。

若顏色亂成一團或質心偏離中心，通常代表 k 選得不對或特徵尺度有問題。

In [ ]:
# 概念：依 labels_ 上色、疊上 cluster_centers_，一眼判讀分群品質
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

rng = np.random.default_rng(0)
center_a = rng.normal(loc=[2, 2], scale=0.5, size=(30, 2))
center_b = rng.normal(loc=[8, 3], scale=0.5, size=(30, 2))
center_c = rng.normal(loc=[5, 8], scale=0.5, size=(30, 2))
points = np.vstack([center_a, center_b, center_c])

kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = kmeans.fit_predict(points)
centers = kmeans.cluster_centers_

plt.figure(figsize=(5, 4))
# c=labels 讓同群同色；cmap 指定配色，不同群自動拿到不同顏色
plt.scatter(points[:, 0], points[:, 1], c=labels, cmap="viridis")
plt.scatter(centers[:, 0], centers[:, 1], color="red", marker="X", s=200)   # 質心用紅叉標出
plt.title("分群結果：依群別上色並標記質心")
plt.xlabel("特徵 1")
plt.ylabel("特徵 2")
plt.show()

# 解讀依據：三群顏色分得開、紅叉落在各團中央，代表分群合理
print("各群點數:", np.bincount(labels))

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）樓高分兩群（整合：分群 + fit_predict + labels_）
#   情境：你有一批建築的樓高（floor height）數值，混有低矮房與高樓兩種，但沒有標好哪棟屬於哪類。
#   要求：
#     1. 用 numpy 造兩團不同高度的樓高資料（例如一團在 10 公尺附近、一團在 50 公尺附近），合併成一份一維資料。
#     2. 把資料 reshape 成 (N, 1)，用 KMeans 設 n_clusters=2 做 fit_predict 取得 labels_，並印出各群點數與兩個質心。
#   驗收：應該看到資料被分成「低樓群」與「高樓群」兩群，兩個質心分別落在低、高樓高附近，分界大致符合直覺。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）街區分群與群數選擇（整合：質心迭代直覺 + inertia + 肘部法 + cluster_centers_）
#   情境：你為每個街區（block）記錄「樓高、基地面積」兩維資料，城市裡有三種型態的街區，但沒有人事先分類。
#   要求：
#     1. 用 numpy 造三團二維街區資料（各自圍繞不同中心）並合併。
#     2. 對 k 從 1 到 6 各跑一次 KMeans，記錄每個 inertia（用 kmeans.inertia_）。
#     3. 畫肘部法曲線挑出合理 k，再用該 k 重新 fit 一次取得 cluster_centers_。
#   驗收：應該看到 inertia 曲線在 k=3 附近出現手肘，且三個質心分別落在三團資料中央。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）容積率分群的尺度陷阱（整合：分群 + inertia + 視覺化 + 特徵尺度思考）
#   情境：你為街區記錄「樓高（數十公尺）」與「容積率 floor area ratio（小數，約 0 到 5）」兩維資料，
#         兩個特徵的數值範圍差很大。
#   要求：
#     1. 用 numpy 造這兩維資料（樓高量級為數十、容積率為小數），直接對原始資料做 KMeans 並依群別上色視覺化。
#     2. 將兩個特徵各自做 z-score 標準化（減平均、除標準差）後，再做一次 KMeans 並視覺化。
#     3. 比較兩次分群的 labels 與 inertia，並用文字說明為何尺度差異會讓某一維（樓高）主導分群。
#   驗收：應該看到未標準化時分群幾乎只跟著樓高走（界線沿樓高方向切），
#         標準化後兩個特徵才一起影響分群結果。
# TODO: 在這裡完成


## 小結

- 分群屬於非監督學習：資料只有特徵、沒有標籤，模型靠資料結構自己分組，與需要答案的分類形成對照。
- KMeans 的核心是反覆做「把點分到最近質心、再把質心移到群平均」，迭代到質心幾乎不動就收斂，不必硬背公式。
- sklearn 的 KMeans 用 n_clusters 設群數，fit_predict 一次取得分群，labels_ 是每點群編號、cluster_centers_ 是各群質心座標。
- inertia 衡量群內離散程度，會隨 k 變大而單調下降；肘部法看 inertia 對 k 曲線的轉折點挑合理 k。
- 分群沒有正確答案，依群別上色並疊上質心做視覺檢查，看群是否分得開、質心是否落在中央。
- 特徵尺度差很大時，數值大的特徵會主導距離計算與分群結果，先標準化能讓各特徵公平地一起影響分群。